# CarRacing RL Competition — 로컬(Windows) Quickstart

Colab 없이 내 PC에서 학습하고 제출 파일을 만드는 노트북입니다.

- **이 노트북은 `carracing_competition` 폴더(`train.py`가 있는 폴더)에서 여세요.** VS Code나 Jupyter 모두 됩니다.
- 커널은 이 프로젝트용 Python 환경(예: conda 가상환경)을 선택하세요. 아래 명령은 모두 **노트북 커널과 같은 Python**으로 실행됩니다.
- **GPU가 없어도 학습은 됩니다.** 다만 CPU만 쓰면 약 2배 느립니다 (예: 30만 프레임이 GPU로 약 30분이면 CPU로 약 1시간, 노트북 CPU는 더 걸릴 수 있음).
- 셀 실행 중 중지(■) 버튼을 누르면 학습도 멈춥니다. `--resume`으로 이어서 학습할 수 있습니다.

## 0. 준비: 실행 도우미와 환경 점검

In [ ]:
import os, sys, subprocess, importlib.metadata as md

assert os.path.exists("train.py"), "이 노트북을 carracing_competition 폴더(train.py가 있는 폴더)에서 여세요."


def run(*args):
    """Run a project script with the same Python as this notebook and stream its output here."""
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8", "PYTHONWARNINGS": "ignore"}
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace", env=env)
    try:
        for line in p.stdout:
            print(line, end="")
        if p.wait() != 0:
            print(f"\n(exit code {p.returncode})")
    except KeyboardInterrupt:
        p.terminate()
        print("\n중지됨 (이어서 학습하려면 같은 셀을 다시 실행하세요)")


print("Python", sys.version.split()[0], "|", sys.executable)
PINNED = {"gymnasium": "1.2.2", "box2d": "2.3.10", "pygame": "2.6.1", "onnx": "1.23.0", "onnxruntime": "1.23.2"}
for name, want in PINNED.items():
    try:
        have = md.version(name)
    except md.PackageNotFoundError:
        have = None
    print(f"  {name:12s} {have or '(없음)':10s} {'OK' if have == want else '<- ' + want + ' 필요'}")
try:
    import torch
    print("  torch       ", torch.__version__, "| GPU:", torch.cuda.is_available())
except ImportError:
    print("  torch 없음 -> 1번 셀로 설치")

## 1. 패키지 설치 (처음 한 번)
위 점검에서 `<- ... 필요`가 보이면 실행하세요. 설치 후에는 **커널을 다시 시작**하고 0번 셀부터 다시 실행합니다.

In [ ]:
%pip install -q -r requirements.txt pandas matplotlib

## 2. 학습
- `QUICK_TEST = True`: 2만 프레임으로 흐름만 확인 (몇 분). 점수는 무작위 수준입니다.
- `QUICK_TEST = False`: 대회 예산 30만 프레임 (PC에 따라 약 40분~2시간)
- 같은 셀을 다시 실행하면 `--resume`으로 이어서 학습합니다. 새로 학습하려면 `RUN_DIR`을 바꾸세요.

In [ ]:
AGENT = "dqn"          # "dqn" / "a2c" / agents/ 폴더에 추가한 에이전트 이름
SEED = 0
QUICK_TEST = True      # True: 2만 프레임 / False: 30만 프레임 (대회 예산)
EXTRA_CFG = []         # 예: ["lr=3e-4", "batch_size=128"]  (A2C 연속 행동: ["continuous=True"])

RUN_DIR = f"runs/{AGENT}_s{SEED}" + ("_quick" if QUICK_TEST else "")
args = ["train.py", "--agent", AGENT, "--seed", str(SEED), "--run-dir", RUN_DIR, "--resume"]
cfg = list(EXTRA_CFG)
if QUICK_TEST:
    args += ["--budget", "20000", "--eval-every", "10000"]
    if AGENT == "dqn":
        cfg.append("learning_starts=500")
if cfg:
    args += ["--cfg", *cfg]
run(*args)

## 3. 학습 곡선

In [ ]:
import csv
import pandas as pd
import matplotlib.pyplot as plt

with open(f"{RUN_DIR}/train.csv") as f:
    rows = list(csv.reader(f))
train = pd.DataFrame([r[:5] for r in rows[1:]], columns=rows[0][:5]).astype({"frames": int, "return": float})
ev = pd.read_csv(f"{RUN_DIR}/eval.csv")
plt.plot(train.frames, train["return"].rolling(10, min_periods=1).mean(), label="train (10-ep avg)")
plt.plot(ev.frames, ev.score, "o-", label="eval")
plt.xlabel("frames"); plt.ylabel("return"); plt.legend(); plt.show()

## 4. 평가 및 주행 영상

In [ ]:
run("evaluate.py", "--ckpt", f"{RUN_DIR}/best.pt", "--seeds", "1000-1002", "--video", f"videos/{AGENT}")

In [ ]:
import glob
from IPython.display import Video

videos = sorted(glob.glob(f"videos/{AGENT}/*.mp4"), key=os.path.getmtime)
Video(videos[-1], embed=True, width=400) if videos else print("영상 파일이 없습니다. moviepy가 설치되어 있는지 확인하세요.")

## 5. 리더보드 제출 파일 만들기
`TEAM_NAME`을 팀 이름으로 바꾸세요 (파일 이름이 팀 이름으로 시작해야 합니다). 행동 일치율이 95% 미만이면 제출하지 마세요.

만들어진 `<팀이름>.onnx`를 Google Classroom 과제에 제출하면 됩니다.

In [ ]:
TEAM_NAME = "MyTeam"
run("export.py", "--ckpt", f"{RUN_DIR}/best.pt", "--out", f"{TEAM_NAME}.onnx")
run("check_submission.py", "--onnx", f"{TEAM_NAME}.onnx")